In [ ]:
import os
WORK_DIR = 'Your local path'
os.chdir(WORK_DIR)

TEST_JSON_PATH = "Sub_Task/cosmos_anns_acm/cosmos_anns_acm/acm_anns/public_test_acm.json"
# Intermediate results save path
SEARCH_INFO_PATH = "Sub_Task/cosmos_anns_acm/cosmos_anns_acm/acm_anns/public_test_acm_search_withbrowser.json"
QWEN_PATH1 = "Sub_Task/cosmos_anns_acm/cosmos_anns_acm/acm_anns/public_test_acm_search_vl_search.json"
QWEN_PATH2 = "Sub_Task/cosmos_anns_acm/cosmos_anns_acm/acm_anns/public_test_acm_search_vl_prompt.json"
INTERN_PATH1 = "Sub_Task/cosmos_anns_acm/cosmos_anns_acm/acm_anns/public_test_acm_internvl_prompt.json"
VOTE_PATH = 'public_final_pred.json'

print(os.path.exists(TEST_JSON_PATH))

In [ ]:
import os

# local model path
deepresearcher_path = "Sub_Task/Models/DeepResearcher-7b"
qwenvl_path = "Sub_Task/Models/Qwen2.5-VL-72B-Instruct-AWQ"
internvl_path = "Sub_Task/Models/InternVL3-78B-AWQ"

# download model form HF
os.system(f'huggingface-cli download GAIR/DeepResearcher-7b --local-dir {deepresearcher_path}')
os.system(f'huggingface-cli download Qwen/Qwen2.5-VL-72B-Instruct-AWQ --local-dir {qwenvl_path}')
os.system(f'huggingface-cli download OpenGVLab/InternVL3-78B-AWQ --local-dir {internvl_path}')

In [ ]:
import os
import sys
import json
import time
import io
import uuid
import subprocess
import requests
from transformers import AutoConfig, AutoModelForCausalLM, AutoModelForTokenClassification, AutoTokenizer
import yaml
import glob
import os
import json5
from tqdm import tqdm
from openai import OpenAI

import re
from tools import *
from smolagents import OpenAIServerModel

model = OpenAIServerModel(
    model_id="factmodel",
    api_base="http://localhost:3001/v1", # Leave this blank to query OpenAI servers.
    api_key="123", # Switch to the API key for the server you're targeting.
    flatten_messages_as_text=False, 
    reasoning_effort="low",
    temperature=0,  
)

modelname = 'Sub_Task/Models/DeepResearcher-7b'
tokenizer = AutoTokenizer.from_pretrained(modelname)

llmclient = OpenAI(
    base_url="http://localhost:3001/v1", 
    api_key="123",
)


system_prompt = r'''## Background information 
* You are Deep AI Research Assistant

The question I give you is a complex question that requires a *deep research* to answer.

I will provide you with tools to help you answer the question:
- web_search: Search the web for relevant information from google. You should use this tool if the historical page contentis not enough to answer the question. Or last search result is not relevant to the question.
- browse_webpage: Browse the webpage and return the content that not appeared in the conversation history. You should use this tool if the last action is search and the search result maybe relevant to the question.

You don't have to answer the question now, but you should first think about the research plan or what to search next.

Your output format should be one of the following two formats:

<think>
YOUR THINKING PROCESS
</think>
<answer>
YOUR ANSWER AFTER GETTING ENOUGH INFORMATION
</answer>

or

<think>
YOUR THINKING PROCESS
</think>
<tool_call>
YOUR TOOL CALL WITH CORRECT FORMAT
</tool_call>

You should always follow the above two formats strictly.
Only output the final answer (in words, numbers or phrase) inside the <answer></answer> tag, without any explanations or extra information. If this is a yes-or-no question, you should only answer yes or no.


# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
{"type": "function", "function": {"name": "web_search", "description": "Search the web for relevant information from google. You should use this tool if the historical page content is not enough to answer the question. Or last search result is not relevant to the question.", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "The query to search, which helps answer the question"}}, "example": {"name": "web_search", "arguments": {"query": "xxxx"}}, "uniqueItems": true}}}
{"type": "function", "function": {"name": "browse_webpage", "description": "Browse the webpage and return the content that not appeared in the conversation history. You should use this tool if the last action is search and the search result maybe relevant to the question.", "parameters": {"type": "object", "properties": {"url_list": {"type": "array", "description": "The chosen urls from the search result, do not use url that not appeared in the search result. do not use more than 3 urls."}, "query": {"type": "string", "description": "These queries aim to retrieve information from the URL webpage."}}, "example": {"name": "browse_webpage", "arguments": {"url_list": ["http://www.dada.com", "xxxx"], "query": "xxxx"}}, "uniqueItems": true}}}

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>'''



def parse_response(response_contents, think: bool = True):
    """Parse response to get the thinking process and answer or tool call.
        return: [(is_stop, thinking, answer/tool_call), ...]
    """
    results = []
    for i, content in enumerate(response_contents):
        if "<think>" in content and "<answer>" in content:
            if "</think>" not in content or "</answer>" not in content:
                results.append((True, "", ""))
            else:
                think = content.split("<think>")[1].split("</think>")[0]
                answer = content.split("<answer>")[1].split("</answer>")[0]
                results.append((True, think, answer))
        elif "<think>" in content and "<tool_call>" in content:
            if "</tool_call>" not in content or "</think>" not in content:
                results.append((True, "", ""))
            else:
                think = content.split("<think>")[1].split("</think>")[0]
                tool_call = content.split("<tool_call>")[1].split("</tool_call>")[0]
                try:
                    tool_call = json.loads(tool_call)
                    results.append((False, think, tool_call))
                except Exception as e:
                    print(f"model tool call format error: {e}")
                    print(i, content)
                    results.append((True, "", ""))
        else:
            results.append((True, "", ""))
    return results


def execute_predictions(
        tool_call_list, total_number
    ) :
    
    query_contents = [{"idx": tool_call[0], "question": tool_call[1], "think": tool_call[2],
                       "tool_call": tool_call[3], "total_number":total_number} for tool_call in tool_call_list]
    tool_result = []
    for query_content in query_contents:
        result = ''
        try:
            if query_content['tool_call']["name"] == 'web_search':
                result = serper_data_serp_api(query_content['tool_call']['arguments']['query'])
            elif query_content['tool_call']["name"] == 'browse_webpage':
                l = []
                for url in query_content['tool_call']['arguments']['url_list']:
                    l.append(web_browser(url,
                                query_content['tool_call']['arguments']['query'],
                                 model,
                                 8000))
                result = l
        except:
            import traceback
            print(f"{str(traceback.format_exc())}")
        tool_result.append(result)
    return tool_result


def run_prompt(question):

    ii = 0
    isfinish = 0
    messages_list = []
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question}
    ]
    messages_list.append(messages)
    result = []
    history = ''
    refs = []
    while(ii < 5):
        rollings_active = tokenizer.apply_chat_template([messages], add_generation_prompt=True, tokenize=False)
        completion = llmclient.completions.create(
            model="factmodel",
            prompt=rollings_active,
            temperature=0.0,
            max_tokens=2024,
        )
        response = json.loads(completion.model_dump_json())['choices'][0]['text']
        results = parse_response([response])[0]

        tool_call_list = []
        if results[0]:
            messages.append({"role": "user", "content": response})
            result.append(response)
            isfinish = 1
            break
        else:
            tool_call_list.append((0, question, results[1], results[2]))
        tool_result = execute_predictions(tool_call_list,1)
        messages.append(
            {
                "role": "assistant", 
                "content": "<think>" + tool_call_list[0][2] + "</think>", 
                "tool_calls": [
                                {
                                    "type": "function", 
                                    "function": tool_call_list[0][3]
                                }
                            ]
            }
        )
        messages.append(
            {
                "role": "tool", 
                "name": tool_call_list[0][3]["name"],
                "content": tool_result[0]
            }
        )
        result.append(tool_result[0])
        ii += 1
        if isfinish == 1:
            break
            
    return result


results = []
inputs = []
for line in open(TEST_JSON_PATH, 'r'):
    if not line:
        continue
    content = json5.loads(line.strip())
    inputs.append(content)
for e in tqdm(inputs):
    r1 = run_prompt(e['caption1'] + '\nis it Fact? True or False or Unknown')
    results.append({
        'img_local_path':e['img_local_path'],
        'r1':r1,
    })
    
f = open(SEARCH_INFO_PATH,'w')
for e in results:
    print(json.dumps(e),file = f)
f.close()

In [ ]:
import os
os.environ['VLLM_WORKER_MULTIPROC_METHOD']='spawn'
os.environ['VLLM_USE_V1']='0'

from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info
from vllm.multimodal.utils import fetch_image
import base64
import re
import json
import pickle
from tqdm import tqdm
from PIL import Image
import sys

import glob
import json5
def remove_urls(obj):
    if isinstance(obj, dict):
        keys_to_delete = []
        for k, v in obj.items():
            if k == 'url':
                keys_to_delete.append(k)
            else:
                remove_urls(v)
        for k in keys_to_delete:
            del obj[k]
    elif isinstance(obj, list):
        for item in obj:
            remove_urls(item)
            
def file_to_data_url(file_path: str):
    """
    Convert a local image file to a data URL.
    """    
    with open(file_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode('utf-8')
    
    _, extension = os.path.splitext(file_path)
    mime_type = f"image/{extension[1:].lower()}"
    
    return f"data:{mime_type};base64,{encoded_string}"

MODEL_PATH = 'Sub_Task/Models/Qwen2.5-VL-72B-Instruct-AWQ'

llm = LLM(
    model=MODEL_PATH,
    limit_mm_per_prompt={"image": 2},
    tensor_parallel_size=1,
    gpu_memory_utilization=0.95,
    dtype='float16',
    max_model_len=16096,
)
processor = AutoProcessor.from_pretrained(MODEL_PATH)
results = []
inputs = []
for line in open(TEST_JSON_PATH,'r'):
    if not line:
        continue
    content = json5.loads(line.strip())
    inputs.append(content)
i= 0
for line in open(SEARCH_INFO_PATH,'r'):
    if not line:
        continue
    content = json.loads(line.strip())
    remove_urls(content['r1'])
    answer_match = re.search(r'<answer>\s*(.*?)\s*</answer>', str(content['r1'][-1]), re.DOTALL)
    answer = answer_match.group(1) if answer_match else None
    content['r1'] = content['r1'][:-1]
    content['r1_answer'] = answer
    inputs[i].update(content)
    i += 1
    
messages = []
for e in tqdm(inputs):
    filename='Sub_Task/images_test_acm/' + e['img_local_path']
    caption = str(e['caption1'])
    search = str(e['r1'])[-20000:]
    prompt = f'''You are given a news image, a news caption, and relevant information searched on the internet as input. 
The news caption is {caption}
Relevant information is {search}

Your task is to determine whether this image-caption pair is **genuine (real)** or **falsely generated (fake)**.

### Input:
1. **Image**: An image provided as visual input.
2. **Caption**: A textual description that supposedly describes the content of the image.
3. **Relevant Information**: Some supporting information retrieved from the web or metadata that might help verify the authenticity of the image-caption pair.

### Task:
Based on your understanding of the image, the caption, and any inconsistencies or clues you observe in conjunction with the relevant information, classify the pair into one of two categories:
0 if the pair is **real** (the caption accurately and naturally describes the image).
1 if the pair is **fake** (the caption does not match the image or appears artificially created to mislead).

### Instructions:
- Analyze the visual content of the image carefully.
- Evaluate whether the caption accurately reflects what is shown in the image.
- Look for contradictions, logical mismatches, or implausible scenarios between the image and the caption.
- Use the provided relevant information (e.g., facts, dates, locations, object details) to cross-check and validate the consistency of the caption with the image.
- If there are signs of image manipulation, caption hallucination, or context mismatch, consider it a fake pair.
- 如果caption中没有出现人名/地名/公司名/新闻等实际信息，只要判断图文是不是匹配/是否有aigc可能就行，不需要关注relevant information，如果匹配则为 real，否则则为fake。

### Example:
Input:
- Image: [A photo showing a sunset over the ocean]
- Caption: "A beautiful sunrise over the Rocky Mountains"
- Relevant Information: "The image EXIF data shows GPS coordinates pointing to a beach in California"

# 格式使用json格式,以下是格式规范：
# ```json
# {{
#         "answer":"0/1"
# }}'''
    messages.append([filename, prompt])
    

messages=[[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image":fetch_image(file_to_data_url(messages[i][0])),
                    "min_pixels": 1000000,
                    "max_pixels": 1000000,
                },
                {
                    "type": "text", 
                    "text": messages[i][1],
                },
            ],
        },
] for i in range(len(messages))
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
image_inputs, video_inputs = process_vision_info(messages)

mm_ls = []
for i in range(len(prompt)):
    mm_data = {}
    if image_inputs is not None:
        mm_data["image"] = image_inputs[i]
    mm_ls.append(mm_data)

llm_inputs = [{
    "prompt": prompt[i],
    "multi_modal_data": mm_ls[i],
} for i in range(len(prompt))]
 
sampling_params = SamplingParams(
    temperature=0.1,
    top_p=1.0,
    repetition_penalty=1.00,
    max_tokens=8096,
    stop_token_ids=[],
)


outputs = llm.generate(llm_inputs, sampling_params=sampling_params)
outputs = [x.outputs[0].text for x in outputs]
for output in outputs[:5]:
    print('----example-----')
    print(output)
    print('----example end-----')

f = open(QWEN_PATH2,'w')
for e in outputs:
    print(json.dumps(e),file = f)
f.close()

messages = []
for e in tqdm(inputs):
    filename='Sub_Task/images_test_acm/' + e['img_local_path']
    caption = str(e['caption1'])
    search = str(e['r1'])[-20000:]
    prompt = f'''给出的图像和caption是真的吗？你可以参考我在搜索引擎搜到的结果。最后请回答True 或 False 或 Unknown。
search result：{search}
caption：{caption}
    
格式使用json格式,以下是格式规范：
```json
{{
        "analysis":"<分析过程>",
        "answer":"<True/False/Unknown>"
}}
```
       

- 如果文本和图像是伪造的则回答 False。
- 如果文本和图像不匹配则回答 False。
- 如果根据常识能判断文本和图像是客观真实的，则回答 True。
- 假如文本是一个客观新闻事实，根据常识没有办法判断文本和图像是不是客观真实的，则回答 Unknown。
- 年份/日期相关的信息不能证明是False，比如next day/month/year等。
    '''
    messages.append([filename, prompt])
    

messages=[[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image":fetch_image(file_to_data_url(messages[i][0])),
                    "min_pixels": 1000000,
                    "max_pixels": 1000000,
                },
                {
                    "type": "text", 
                    "text": messages[i][1],
                },
            ],
        },
] for i in range(len(messages))
]

prompt = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
image_inputs, video_inputs = process_vision_info(messages)

mm_ls = []
for i in range(len(prompt)):
    mm_data = {}
    if image_inputs is not None:
        mm_data["image"] = image_inputs[i]
    mm_ls.append(mm_data)

llm_inputs = [{
    "prompt": prompt[i],
    "multi_modal_data": mm_ls[i],
} for i in range(len(prompt))]
 
sampling_params = SamplingParams(
    temperature=0.1,
    top_p=1.0,
    repetition_penalty=1.00,
    max_tokens=8096,
    stop_token_ids=[],
)


outputs = llm.generate(llm_inputs, sampling_params=sampling_params)
outputs = [x.outputs[0].text for x in outputs]
for output in outputs[:5]:
    print('----example-----')
    print(output)
    print('----example end-----')

f = open(QWEN_PATH1,'w')
for e in outputs:
    print(json.dumps(e),file = f)
f.close()


In [ ]:
import os
os.environ['VLLM_WORKER_MULTIPROC_METHOD']='spawn'

from vllm import LLM, SamplingParams
from PIL import Image
import json
# import json5

# modify mllm model path
model_path = "Sub_Task/Models/InternVL3-78B-AWQ"
llm = LLM(
    model=model_path, 
    tensor_parallel_size=1,
    gpu_memory_utilization=0.95,
    quantization='awq',
    dtype='float16',
    max_model_len=8192
)

# json path
test_json_path = TEST_JSON_PATH
search_json_path = SEARCH_INFO_PATH

def remove_urls(obj):
    if isinstance(obj, dict):
        keys_to_delete = []
        for k, v in obj.items():
            if k == 'url' or k == 'finish' or k == 'workpath':
                keys_to_delete.append(k)
            else:
                remove_urls(v)
        for k in keys_to_delete:
            del obj[k]
    elif isinstance(obj, list):
        for item in obj:
            remove_urls(item)

# read data
all_samples = {}
with open(test_json_path, "r") as f:
    for line in f.readlines():
        data = json.loads(line)
        all_samples[data["img_local_path"]] = [data["caption1"], data["context_label"]]

with open(search_json_path, "r") as f:
    for line in f.readlines():
        data = json.loads(line)
        r1 = data["r1"]
        remove_urls(r1)
        if len(r1) > 1:
            r1 = r1[:-1]
        all_samples[data["img_local_path"]].append(r1)

prompt = """You are given a news image, a news caption, and relevant information searched on the internet as input. 
The news caption is {caption}
The new image is <image>
Relevant information is {evidence}

Your task is to determine whether this image-caption pair is **genuine (real)** or **falsely generated (fake)**.

### Input:
1. **Image**: An image provided as visual input.
2. **Caption**: A textual description that supposedly describes the content of the image.
3. **Relevant Information**: Some supporting information retrieved from the web or metadata that might help verify the authenticity of the image-caption pair.

### Task:
Based on your understanding of the image, the caption, and any inconsistencies or clues you observe in conjunction with the relevant information, classify the pair into one of two categories:
0 if the pair is **real** (the caption accurately and naturally describes the image).
1 if the pair is **fake** (the caption does not match the image or appears artificially created to mislead).

### Instructions:
- Analyze the visual content of the image carefully.
- Evaluate whether the caption accurately reflects what is shown in the image.
- Look for contradictions, logical mismatches, or implausible scenarios between the image and the caption.
- Use the provided relevant information (e.g., facts, dates, locations, object details) to cross-check and validate the consistency of the caption with the image.
- If there are signs of image manipulation, caption hallucination, or context mismatch, consider it a fake pair.
- Output only the class label (0 or 1) based on your analysis.

### Example:
Input:
- Image: [A photo showing a sunset over the ocean]
- Caption: "A beautiful sunrise over the Rocky Mountains"
- Relevant Information: "The image EXIF data shows GPS coordinates pointing to a beach in California"

Output:
1

Now, analyze the provided image-caption pair and relevant information, output your prediction.
You should answer in the following form: '0' or '1'.
The answer is:
"""

# image root
image_root = "Sub_Task/images_test_acm/"

# modify save_path
save_path = INTERN_PATH1

params = SamplingParams(
    temperature=0,
    top_p=1.0,
    repetition_penalty=1.0,
    max_tokens=8192,
)

def request_mllm(image_path, input_prompt):
    image = Image.open(image_path)
    response = llm.generate(
        {
            "prompt": input_prompt,
            "multi_modal_data": {"image": image},
        },
        sampling_params=params,
        use_tqdm=False
    )
    answer = response[0].outputs[0].text
    return answer.strip().strip(",.，。")

# test 
for local_path in all_samples:
    image_path = os.path.join(image_root, local_path)
    caption, label, evidence = all_samples[local_path]
    curr_prompt = prompt.format(caption=caption, evidence=evidence)
    response = request_mllm(image_path, curr_prompt)
    print("test output:", response, "label:", label)
    break

# inference
with open(save_path, "w", encoding="utf-8") as f:
    for local_path in all_samples:
        image_path = os.path.join(image_root, local_path)
        caption, label, evidence = all_samples[local_path]
        curr_prompt = prompt.format(caption=caption, evidence=evidence)
        answer = request_mllm(image_path, curr_prompt)
        record = {
            "img_local_path": local_path,
            "gt": label,
            "pred": answer
        }
        record_line = json.dumps(record)
        f.write(f"{record_line}\n")
        f.flush()
f.close()


In [ ]:
import json5,json,re

def extract_json_from_text(text):
    data = '' 
    
    try:
        if not isinstance(text, str):
            raise TypeError("The input must be of string type.")

        pattern = re.compile(r'\{.*?\}', re.DOTALL)
        matches = pattern.finditer(text)
        
        last_match = None
        for match in matches:
            last_match = match

        if last_match:
            json_str = last_match.group()
            try:
                data = json.loads(json_str)
            except json.JSONDecodeError as e:
                print(f"JSON parsing error: {e}")
        else:
            print("No JSON data found")

    except TypeError as te:
        print(f"Type error: {te}")
    except Exception as e:
        print(f"Unexpected error: {e}")

    return data


gts = []
preds = []
preds_hfprompt = []
preds_hanjian = []

# 用hanjian的prompt
for line in open(QWEN_PATH2,'r'):
    if not line:
        continue
    content = extract_json_from_text(json.loads(line.strip()))
    preds_hfprompt.append('True' if int(content['answer']) == 0 else 'False')
    
# 用自己的prompt
for line in open(QWEN_PATH1,'r'):
    if not line:
        continue
    content = extract_json_from_text(json.loads(line.strip()))
    preds.append(content['answer'])

# hanjian的文件
for line in open(INTERN_PATH1,'r'):
    if not line:
        continue
    content = json.loads(line.strip())
    preds_hanjian.append('True' if int(content['pred']) == 0 else 'False')
    gts.append(content['gt'])
    
# 融合
def combine(l):
    vc = {}
    for e in l:
        vc[e] = vc.get(e,0) + 1
    if vc.get('True',0) > 0 and vc.get('True',0) > vc.get('False',0):
        return 'True'
    if vc.get('False',0) > 0 and vc.get('False',0) > vc.get('True',0):
        return 'False'
    return 'False'
    
# input_data = inputs
preds = [combine(x) for x in zip(preds,preds_hfprompt,preds_hanjian)]

f = open(VOTE_PATH,'w')
i = 0
for line in open(TEST_JSON_PATH,'r'):
    if not line:
        continue
    content = json5.loads(line.strip())
    content['pred'] = preds[i]
    print(json.dumps(content),file = f)
    i += 1
f.close()

In [21]:
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
import numpy as np

def calculate_metrics(y_true, y_pred):
    """
    compute ACC, Precision, Recall, F1
    """
    try:
        acc = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred, pos_label=1)
        recall = recall_score(y_true, y_pred, pos_label=1)
        f1 = f1_score(y_true, y_pred, pos_label=1)
        mcc = matthews_corrcoef(y_true, y_pred)
        
        metrics = {
            'Accuracy': round(acc, 6),
            'Precision': round(precision, 6),
            'Recall': round(recall, 6),
            'F1-score': round(f1, 6),
            'MCC': round(mcc, 6)
        }
        
        return metrics
    
    except Exception as e:
        print(f"Error occurred: {str(e)}")
        return None

final_preds = []
with open(VOTE_PATH, 'r') as f:
    for line in f:
        content = json.loads(line)
        pred = 0 if content['pred'] == 'True' else 1
        final_preds.append(pred)

metrics = calculate_metrics(gts, final_preds)
for key in metrics:
    print(key, metrics[key])
        

Accuracy 0.91
Precision 0.91
Recall 0.91
F1-score 0.91
MCC 0.82
